# Phase B - the quantum-adapter grid

Input: the three `features_*.parquet` tables and the spatial maps Phase A wrote.
Run cells 1 -> 15 in order. Cell 4 loads them and takes the intersection of the
three cohorts (3,889 series) - that is what every comparison runs on, so no
difference between backbones can be a difference between cohorts.

The grid is 3 backbones x 4 quantum families = 12 cells on `any`, resumable per
cell. Cell 8 benchmarks the differentiation method on your machine before the
grid starts, so check that number rather than walking away immediately.


In [ ]:
# 0. Repository and paths -- every location is read from config.yaml (see README), none is typed here.
import os
import sys


def _find_repo():
    for c in (os.environ.get("QCT3D_REPO", ""), os.getcwd(), os.path.dirname(os.getcwd()),
              "/content/qct3d-saturation"):
        if c and os.path.isfile(os.path.join(c, "src", "qct3d", "config.py")):
            return os.path.abspath(c)
    raise RuntimeError("Cannot find the repository. Clone it first (README, 'Running on Colab') "
                       "or set the environment variable QCT3D_REPO to its folder.")


REPO_ROOT = _find_repo()
if os.path.join(REPO_ROOT, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
from qct3d.config import load_config  # noqa: E402

P = load_config()

In [ ]:
# 1. Drive
import os

try:
    from google.colab import drive
    drive.mount(P["drive_mount"])
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("not in Colab -- point OUTPUT_DIR at wherever Phase A wrote its caches.")
if IN_COLAB and not os.path.isdir(P["drive_root"]):
    raise RuntimeError("Drive did not finish mounting -- re-run this cell.")
print("ok")

In [ ]:
# 2. Installs
#
# Phase B needs far less than Phase A: no MONAI, no pydicom, no SimpleITK. Only PennyLane
# (the circuits), torch (the graph they live in), and pyarrow (to read the feature tables).
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pennylane>=0.35,<0.50", "pyarrow>=14", "pandas>=2.1", "scikit-learn>=1.3"],
               check=False)

import pennylane as qml
import torch

print(f"pennylane {qml.__version__}   torch {torch.__version__}")
print("NOTE: this notebook does not need a GPU. A CPU runtime is the right choice --")
print("      it leaves your GPU quota for Phase A, where it actually matters.")

In [ ]:
# 3. CONFIGURATION
import os

# Must match Phase A.
OUTPUT_DIR   = P["rsna_output_dir"]
FEATURE_DIR  = os.path.join(OUTPUT_DIR, "features")
SPATIAL_DIR  = os.path.join(OUTPUT_DIR, "spatial_maps")

# ---- the grid -------------------------------------------------------------------------
# Comment out any backbone you have not extracted yet; the notebook reports what it found
# and runs the rest rather than failing.
BACKBONES_TO_RUN = [
    "B1_swinunetr_suprem",
    "B2_segresnet_suprem",
    "B3_resnet18_medicalnet",
    # "B4_swinunetr_ssl5050",
]
FAMILIES_TO_RUN = ["Q1_sel", "Q2_reupload", "Q3_quanv", "Q4_ttn"]

# ---- the reducer ----------------------------------------------------------------------
# How a 768-2304-d feature vector becomes N_QUBITS rotation angles matters MORE than the
# choice of ansatz. Running one and presenting it as neutral is the easiest way to mislead
# yourself, so the grid runs on the primary and the finalists are re-run on all three.
#
#   "pca_supervised"  PCA on the training fold, then the components most correlated with
#                     the label. Nothing outside the adapter is trainable.
#   "trainable"       a trainable Linear(d -> n_qubits) trained jointly with the circuit.
#                     Highest performing; holds most of the parameters, which is exactly
#                     why the classical control shares it and only the TAIL is matched.
#   "pca"             plain top-variance PCA. Label-blind, and the honest floor.
PRIMARY_REDUCER  = "pca_supervised"
FINALIST_REDUCERS = ["pca_supervised", "trainable", "pca"]

# ---- targets --------------------------------------------------------------------------
PRIMARY_TARGET   = "any"          # ICH presence: the powered endpoint
SECONDARY_TARGETS = ["intraparenchymal", "subdural", "subarachnoid", "intraventricular",
                     "epidural"]
RUN_SECONDARY_ON_FINALIST_ONLY = True    # subtypes are exploratory; see the outline
MIN_POSITIVES    = 30

# ---- circuit --------------------------------------------------------------------------
N_QUBITS   = 4       # must be a power of two: Q4's tree ansatz needs one
DEPTH      = 3       # SEL layers (Q1/Q3) and re-uploads (Q2); Q4's depth is fixed by the tree
DIFF_METHOD = "backprop"      # measured against adjoint in the benchmark cell below
INIT_STD   = 0.01             # near-identity init: barren-plateau mitigation

# ---- optimisation ---------------------------------------------------------------------
EPOCHS          = 80
EPOCHS_SPATIAL  = 25      # Q3 sees n_positions times more circuit evaluations per epoch
BATCH           = 128
BATCH_SPATIAL   = 32
LR              = 0.05
ANGLE_L2_BETA   = 1e-4    # penalty on the ROTATION ANGLES specifically
PROJ_WEIGHT_DECAY = 1e-3  # on the reducer/projection only
FOCAL_ALPHA, FOCAL_GAMMA = 0.25, 2.0

# ---- protocol ---------------------------------------------------------------------------
CV_FOLDS      = 5
HOLDOUT_FRAC  = 0.2
TARGET_SENSITIVITY = 0.85
N_BOOTSTRAP   = 2000
SEED          = 12345
NISQ_P        = 0.01
FORCE_RERUN   = False

RESULTS_JSONL = os.path.join(OUTPUT_DIR, "phaseb_results.jsonl")
RESULTS_CSV   = os.path.join(OUTPUT_DIR, "phaseb_leaderboard.csv")
FIG_DIR       = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

if N_QUBITS & (N_QUBITS - 1):
    raise SystemExit(f"N_QUBITS must be a power of two (Q4's tree ansatz halves the wires "
                     f"at every level); got {N_QUBITS}")

CFG = dict(n_qubits=N_QUBITS, depth=DEPTH, diff_method=DIFF_METHOD, init_std=INIT_STD,
           epochs=EPOCHS, epochs_spatial=EPOCHS_SPATIAL, batch=BATCH,
           batch_spatial=BATCH_SPATIAL, lr=LR, angle_l2_beta=ANGLE_L2_BETA,
           projection_weight_decay=PROJ_WEIGHT_DECAY, focal_alpha=FOCAL_ALPHA,
           focal_gamma=FOCAL_GAMMA, cv_folds=CV_FOLDS, holdout_frac=HOLDOUT_FRAC,
           seed=SEED, n_bootstrap=N_BOOTSTRAP, target_sensitivity=TARGET_SENSITIVITY,
           predict_batch=256, dataset="rsna-ich")
CFG["signature"] = "|".join([
    f"q={N_QUBITS}", f"d={DEPTH}", f"diff={DIFF_METHOD}", f"ep={EPOCHS}/{EPOCHS_SPATIAL}",
    f"lr={LR}", f"bs={BATCH}/{BATCH_SPATIAL}", f"beta={ANGLE_L2_BETA}",
    f"wd={PROJ_WEIGHT_DECAY}", f"focal=({FOCAL_ALPHA},{FOCAL_GAMMA})",
    f"folds={CV_FOLDS}", f"holdout={HOLDOUT_FRAC}", f"seed={SEED}",
])
print("config signature:", CFG["signature"])
print(f"grid: {len(BACKBONES_TO_RUN)} backbones x {len(FAMILIES_TO_RUN)} families "
      f"= {len(BACKBONES_TO_RUN)*len(FAMILIES_TO_RUN)} cells")

In [ ]:
# 4. Load every backbone's cached features, and the spatial maps the Q3 arm needs
#
# The three backbones do NOT share a feature width (768/128/512 per window x 3 windows),
# which is fine -- each is its own row of the grid and the reducer absorbs the difference.
# What they MUST share is the set of series, because otherwise a "difference between
# backbones" is partly a difference between cohorts. The intersection is taken explicitly
# and reported, rather than each backbone being scored on whatever it happens to have.
import concurrent.futures as _cf
import time as _time

import numpy as np
import pandas as pd

SPATIAL_LOAD_WORKERS = 16     # Drive is latency-bound; this is round trips, not bytes

LABEL_COLS = ["any", "epidural", "intraparenchymal", "intraventricular", "subarachnoid",
              "subdural"]

TABLES, MISSING = {}, []
for b in BACKBONES_TO_RUN:
    p = os.path.join(OUTPUT_DIR, f"features_{b}.parquet")
    if not os.path.exists(p):
        MISSING.append(b)
        continue
    TABLES[b] = pd.read_parquet(p)
    print(f"{b:<26} {len(TABLES[b]):>6,} series x "
          f"{sum(c.startswith('f_') for c in TABLES[b].columns):>5} features")
if MISSING:
    print(f"\nNOT FOUND (skipped): {MISSING}")
    print("  run nb1_phaseA_extract.ipynb with BACKBONE_ID set to each of these first.")
if not TABLES:
    raise SystemExit("no feature tables found -- Phase A has not produced anything yet.")

# The intersection is what protects this notebook from a real hazard in Phase A: cell 15
# there admits any series whose progress record says "ok" under the current preprocessing
# signature, WITHOUT checking that the series is in the current manifest. B1 was extracted
# once before on an earlier 86-series cohort under an identical signature, so 68 of those
# came along into features_B1_swinunetr_suprem.parquet. Because Phase A merges metadata with
# a LEFT join, those rows carry NaN for every label and for `patient` -- and a NaN in
# `patient` would silently corrupt the patient-grouped split that the whole protocol rests
# on. Taking the intersection FIRST removes them; the assertions below prove it.
common = None
for b, t in TABLES.items():
    s = set(t["series"])
    common = s if common is None else (common & s)
common = sorted(common)
print(f"\nseries present in ALL {len(TABLES)} backbone(s): {len(common):,}")
for b, t in TABLES.items():
    if len(t) != len(common):
        _extra = sorted(set(t["series"]) - set(common))
        _nan = int(t[t["series"].isin(_extra)]["any"].isna().sum()) if "any" in t.columns else 0
        print(f"  {b}: dropping {len(t)-len(common):,} series not shared by every backbone"
              + (f"   ({_nan:,} of them have no label at all -- out-of-cohort leftovers)"
                 if _nan else ""))
if len(common) < 50:
    raise SystemExit(f"only {len(common)} shared series -- extract more before running the grid.")

X_BY_BACKBONE, M_BY_BACKBONE = {}, {}
meta = None
for b, t in TABLES.items():
    t = t[t["series"].isin(common)].sort_values("series").reset_index(drop=True)
    fcols = [c for c in t.columns if c.startswith("f_")]
    X_BY_BACKBONE[b] = t[fcols].to_numpy(dtype=np.float32)
    if meta is None:
        meta = t[["series", "patient", "study"] +
                 [c for c in LABEL_COLS if c in t.columns]].copy()

# Every downstream split is grouped by these. A NaN here does not raise -- it becomes the
# string "nan", silently collapsing unrelated patients into ONE group and leaking scans of
# different people across the train/test boundary. Checked rather than trusted.
if meta["patient"].isna().any():
    raise SystemExit(
        f"{int(meta['patient'].isna().sum()):,} of {len(meta):,} rows have no patient id. "
        f"They are series that are in a feature table but not in manifest.parquet, so the "
        f"metadata merge in Phase A left them empty. Every patient-grouped split would be "
        f"wrong. Fix the feature tables (restrict them to manifest.parquet) before running.")
for _lc in [PRIMARY_TARGET] + SECONDARY_TARGETS:
    if _lc in meta.columns and meta[_lc].isna().any():
        raise SystemExit(f"{int(meta[_lc].isna().sum()):,} rows have no {_lc!r} label; "
                         f"same cause and same fix as above.")

SERIES_ALL = meta["series"].to_numpy()
GROUPS_ALL = meta["patient"].astype(str).to_numpy()   # group by PATIENT, never by series
assert len(set(GROUPS_ALL)) > 1, "every series resolved to one patient group"
print(f"patients: {len(set(GROUPS_ALL)):,}  ({len(SERIES_ALL)/max(len(set(GROUPS_ALL)),1):.2f} "
      f"series per patient)")

# ---- spatial maps (Q3 only) -------------------------------------------------------------
NEED_SPATIAL = any(f == "Q3_quanv" for f in FAMILIES_TO_RUN)
if NEED_SPATIAL:
    for b in list(X_BY_BACKBONE):
        d = os.path.join(SPATIAL_DIR, b)
        if not os.path.isdir(d):
            print(f"\nNO SPATIAL MAPS for {b} at {d}")
            print("  Q3 is the deep-placement arm and cannot run from pooled vectors. Either")
            print("  re-run Phase A for this backbone with SAVE_SPATIAL_MAP=True, or drop")
            print("  'Q3_quanv' from FAMILIES_TO_RUN.")
            continue
        # SERIAL LOADING HERE IS THE SAME MISTAKE PHASE A MADE. This is 3,889 individual
        # .npy files per backbone, ~11,700 across three, and on a Drive FUSE mount it is the
        # per-file round trip that costs -- not the bytes. One at a time that is 20 minutes
        # to 3 hours before a single circuit trains; a thread pool makes it a couple of
        # minutes. Order is preserved by index, NOT by completion order: these rows are
        # aligned with meta["series"] and with X_BY_BACKBONE, so shuffling them would pair
        # every spatial map with the wrong label.
        _ids = list(meta["series"])
        _paths = [os.path.join(d, f"{s}.npy") for s in _ids]
        arrs = [None] * len(_paths)

        def _one(i, _p=_paths):
            try:
                return i, np.load(_p[i])
            except Exception:
                return i, None

        _t0 = _time.time()
        with _cf.ThreadPoolExecutor(max_workers=SPATIAL_LOAD_WORKERS) as _ex:
            for _n, (_i, _a) in enumerate(_ex.map(_one, range(len(_paths))), 1):
                arrs[_i] = _a
                if _n % 500 == 0 or _n == len(_paths):
                    print(f"\r  {b}: loaded {_n:,}/{len(_paths):,} maps "
                          f"({_time.time()-_t0:.0f}s)   ", end="", flush=True)
        print()
        _n_missing = sum(1 for _a in arrs if _a is None)
        if _n_missing:
            print(f"  {b}: {_n_missing:,} spatial map(s) missing on disk")
        ok = _n_missing == 0
        if ok and arrs:
            _shapes = {a.shape for a in arrs}
            if len(_shapes) != 1:
                print(f"  {b}: spatial maps have {len(_shapes)} different shapes {list(_shapes)[:3]}"
                      f" -- Q3 skipped for this backbone")
                ok = False
        if ok and arrs:
            M_BY_BACKBONE[b] = np.stack(arrs).astype(np.float32)
            print(f"spatial maps {b:<26} {M_BY_BACKBONE[b].shape}  "
                  f"({M_BY_BACKBONE[b].nbytes/1e6:.0f} MB, "
                  f"{int(np.prod(M_BY_BACKBONE[b].shape[2:]))} positions/series)")
        else:
            print(f"spatial maps {b}: incomplete -- Q3 will be skipped for this backbone")

# ---- targets ----------------------------------------------------------------------------
print(f"\n{'target':<22}{'positives':>11}{'prevalence':>13}{'majority acc':>15}")
TARGETS = {}
for t in [PRIMARY_TARGET] + SECONDARY_TARGETS:
    if t not in meta.columns:
        continue
    y = (meta[t].to_numpy() > 0).astype(int)
    npos = int(y.sum())
    flag = ""
    if npos < MIN_POSITIVES or (len(y) - npos) < MIN_POSITIVES:
        flag = f"  <-- below MIN_POSITIVES={MIN_POSITIVES}, skipped"
    else:
        TARGETS[t] = y
    print(f"  {t:<20}{npos:>11,}{y.mean():>13.4f}{max(y.mean(),1-y.mean()):>15.3f}{flag}")
if PRIMARY_TARGET not in TARGETS:
    raise SystemExit(f"the primary target {PRIMARY_TARGET!r} does not have enough positives.")
print("\n  'majority acc' is what you score by always answering the majority class. Any")
print("  accuracy at or below it is worth nothing, which is why every accuracy below is")
print("  printed next to it.")

In [ ]:
# 5. The four quantum-circuit families, the adapters, and the training loop
#
# Moved verbatim to src/qct3d/models.py and imported here into the notebook namespace.
# Every later cell sees exactly the names this cell used to define.
from qct3d.models import *  # noqa: F401,F403

In [ ]:
# 6. The protocol: patient-grouped splits, the three arms, and the paired comparison
#
# Moved verbatim to src/qct3d/protocol.py and imported here into the notebook namespace.
# Every later cell sees exactly the names this cell used to define.
from qct3d.protocol import *  # noqa: F401,F403

In [ ]:
# 7. Parameter budgets, and a self-test that proves gradients reach the rotation angles
#
# ASSERTED, because each failure means the cell is broken: the angles must MOVE during
# training (if they do not, no gradient is reaching the circuit and every quantum number
# below is an untrained random circuit's output), the loss must go DOWN, and the outputs
# must be finite.
#
# REPORTED, not asserted: the AUROC on a planted signal. How high it gets depends on the
# epoch count, the learning rate and the qubit count, so asserting a floor on it fires
# when nothing is wrong -- an alarm that goes off spuriously is worse than no alarm.
import numpy as np
import torch

print(f"{'family':<14}{'role':<46}{'tail':>6}{'hidden':>8}{'control':>9}{'err':>7}")
print("-" * 92)
BUDGETS = {}
for fam in FAMILIES_TO_RUN:
    probe = AdapterModel("quantum", Reducer("pca", 64, N_QUBITS), N_QUBITS, DEPTH,
                         family=fam, diff_method="backprop")
    tail = probe.tail_params()
    hid, err, cpar = match_hidden_width(N_QUBITS, tail)
    BUDGETS[fam] = {"tail": tail, "hidden": hid, "control_params": cpar, "err": err}
    role = QUANTUM_FAMILIES[fam]["role"]
    print(f"{fam:<14}{role[:44]:<46}{tail:>6}{hid:>8}{cpar:>9}{err:>6.1%}")
    if err > 0.10:
        print(f"  *** {fam}: outside the +/-10% matching band. With this few parameters the "
              f"hidden width is quantised too coarsely -- report the exact counts in the paper "
              f"rather than claiming a match. ***")
print(f"\nlinear probe floor: the full feature vector + 1 bias, no adapter, no reduction")


def _self_test(fam):
    rng = np.random.default_rng(0)
    n, d, k = 600, 128, 20
    L = rng.standard_normal((d, k)).astype("float32")
    Z = rng.standard_normal((n, k)).astype("float32")
    X = np.maximum(Z @ L.T + 0.5 * rng.standard_normal((n, d)), 0).astype("float32")
    s = Z @ rng.standard_normal(k) + rng.standard_normal(n)
    y = (s >= np.quantile(s, 0.86)).astype(int)
    tr, te = np.arange(450), np.arange(450, n)
    red = Reducer("trainable", d, N_QUBITS, seed=0).fit(X[tr], y[tr], seed=0)
    m = AdapterModel("quantum", red, N_QUBITS, DEPTH, family=fam,
                     diff_method=DIFF_METHOD, init_std=INIT_STD, seed=0)
    Ztr, Zte = red.transform_np(X[tr]), red.transform_np(X[te])
    before = m.q.weights.detach().clone()
    Xt = torch.as_tensor(Ztr, dtype=torch.float32)
    yt = torch.as_tensor(y[tr], dtype=torch.float32)
    with torch.no_grad():
        l0 = float(focal_loss(m(Xt), yt, FOCAL_ALPHA, FOCAL_GAMMA))
    train_adapter(m, Ztr, y[tr], epochs=40, lr=LR, batch=BATCH, beta=ANGLE_L2_BETA,
                  weight_decay=PROJ_WEIGHT_DECAY, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, seed=0)
    with torch.no_grad():
        l1 = float(focal_loss(m(Xt), yt, FOCAL_ALPHA, FOCAL_GAMMA))
    moved = float((m.q.weights.detach() - before).abs().max())
    p = predict_proba(m, Zte)
    return moved, l0, l1, fast_auroc(y[te], p), bool(np.isfinite(p).all())


print("\nself-test on synthetic features with a planted signal (~14% prevalence):")
for fam in [f for f in FAMILIES_TO_RUN if not QUANTUM_FAMILIES[f]["spatial"]]:
    mv, l0, l1, auc, fin = _self_test(fam)
    print(f"  {fam:<14} angles moved {mv:.5f} | focal {l0:.5f} -> {l1:.5f} | "
          f"AUROC {auc:.3f} | finite {fin}")
    assert fin, f"{fam}: non-finite probabilities -- the circuit is broken."
    assert mv > 1e-6, (
        f"{fam}: the rotation angles did not move during training (max change {mv:.2e}). No "
        f"gradient is reaching the circuit, so every number this notebook produces for this "
        f"family would be an untrained random circuit's output. Check DIFF_METHOD.")
    assert l1 < l0, (f"{fam}: the focal loss did not decrease ({l0:.5f} -> {l1:.5f}). The "
                     f"optimiser is making no progress; check LR (currently {LR}).")
print("  -> gradients reach the rotation angles and the loss falls: the circuits train.")

In [ ]:
# 8. Differentiation-method benchmark -- measure it, do not inherit it
#
# The usual advice is adjoint over parameter-shift, on the grounds that parameter-shift
# needs 2P forward passes per gradient step. That is correct about parameter-shift and
# incomplete about the alternative.
#
# `default.qubit` is a TORCH-NATIVE statevector simulator. With diff_method="backprop" the
# gradient is taken straight through the simulation by autograd: exact, one backward pass,
# and fully batched across the minibatch. `adjoint` is exact too, but PennyLane's adjoint
# pipeline does not broadcast the same way and falls back to a per-sample loop -- which at
# batch 128 means 128 sequential circuit evaluations where backprop does one batched op.
#
# Nothing about the GRADIENT differs between them; both are exact. Only the cost differs.
# Say exactly that in the methods section.
import time

BENCHMARK_METHODS = ["backprop", "adjoint"]     # add "parameter-shift" for the full set
_Xb = np.random.default_rng(0).standard_normal((256, N_QUBITS)).astype("float32")
_yb = (np.random.default_rng(1).random(256) < 0.14).astype("float32")

print(f"{N_QUBITS} qubits, depth {DEPTH}, 3 epochs over 256 rows at batch {BATCH}\n")
print(f"{'family':<14}{'diff_method':<18}{'seconds':>10}{'x backprop':>13}")
print("-" * 56)
for fam in [f for f in FAMILIES_TO_RUN if not QUANTUM_FAMILIES[f]["spatial"]]:
    base = None
    for dm in BENCHMARK_METHODS:
        try:
            m = AdapterModel("quantum", torch.nn.Identity(), N_QUBITS, DEPTH, family=fam,
                             diff_method=dm, init_std=INIT_STD, seed=0)
            t0 = time.time()
            train_adapter(m, _Xb, _yb, epochs=3, lr=LR, batch=BATCH, beta=ANGLE_L2_BETA,
                          weight_decay=0.0, seed=0)
            dt = time.time() - t0
            base = dt if base is None else base
            print(f"{fam:<14}{dm:<18}{dt:>10.2f}{dt/base:>12.1f}x")
        except Exception as e:
            print(f"{fam:<14}{dm:<18}{'failed':>10}   {type(e).__name__}: {str(e)[:40]}")
print(f"\nconfigured DIFF_METHOD = {DIFF_METHOD!r}")

In [ ]:
# 9. THE GRID  [resumable per (backbone, family, target, reducer)]
#
# Every cell is logged to one table from the first run. Reconstructing this afterwards from
# scattered cell output is not realistically possible, so the logging exists before the
# first result does.
import time

import numpy as np

DONE = set() if FORCE_RERUN else done_cells(RESULTS_JSONL, CFG["signature"])
if FORCE_RERUN:
    print("FORCE_RERUN is on -- every cell is recomputed.")

# ---- self-healing resume -----------------------------------------------------------------
# The resume key only asks "was this cell logged?", never "is what was logged usable?".
# Twice a run was resumed on top of rows that should not have survived, so the check is
# done here instead of relying on anyone remembering to flip FORCE_RERUN.
#
# Two things make a logged cell unusable:
#
#   1. A quantum AUROC of EXACTLY 0.500. That is a constant prediction -- every series
#      scored identically -- not a weak model. Q3 produced exactly this before the
#      feature-map standardisation went in, and resuming past it carries a dead arm
#      straight into the manuscript.
#   2. A fully-logged grid trains nothing, so it leaves no fitted model behind and
#      finalist_adapter.pt can never be written. nb4 attributes that artefact, so if it
#      is missing the primary-endpoint cells have to run again whatever the log says.
_INVALID = set()
if DONE and not FORCE_RERUN:
    _prev = load_results(RESULTS_JSONL, CFG["signature"])
    if not _prev.empty and "holdout_auroc" in _prev:
        _q = _prev[_prev["arm"] == "quantum"]
        # Exact equality to 0.500 was too narrow: it caught B3's Q3 but not B1's and B2's,
        # whose constant predictions landed a hair off 0.5. The real signature of a
        # constant prediction is a DEGENERATE CONFIDENCE INTERVAL -- a patient-clustered
        # bootstrap over 783 held-out series gives a width near 0.07 for a live model and
        # near zero for one that scores everything identically. Width is the test.
        if {"holdout_auroc_hi", "holdout_auroc_lo"}.issubset(_q.columns):
            _narrow = (_q["holdout_auroc_hi"] - _q["holdout_auroc_lo"]) < 0.02
        else:                                   # no CI columns: nothing is narrow
            _narrow = _q["holdout_auroc"] != _q["holdout_auroc"]
        _dead = _q[(~np.isfinite(_q["holdout_auroc"]))
                   | (_q["holdout_auroc"].round(6) == 0.5)
                   | _narrow]
        for _, _r in _dead.iterrows():
            _INVALID.add((_r["backbone"], _r["family"], _r["target"], _r["reducer"]))
        if len(_dead):
            print(f"  {len(_dead)} logged cell(s) look like a CONSTANT prediction "
                  f"(AUROC 0.500, or a bootstrap interval narrower than 0.02). "
                  f"Recomputing them.")
    # A finalist artefact that EXISTS is not the same as one that is CURRENT. The first
    # run exported an exploratory subtype cell; a later run exported the right one. nb4
    # cannot tell the difference, so the sidecar written beside it records which cell and
    # which config signature produced it, and it is checked here.
    _fp = os.path.join(OUTPUT_DIR, "finalist_adapter.pt")
    _fm = os.path.join(OUTPUT_DIR, "finalist_adapter.meta.json")
    _stale = None
    if not os.path.exists(_fp):
        _stale = "finalist_adapter.pt is missing"
    elif not os.path.exists(_fm):
        _stale = "finalist_adapter.pt has no sidecar, so its provenance is unknown"
    else:
        try:
            _m = json.load(open(_fm))
            if _m.get("config_sig") != CFG["signature"]:
                _stale = "the finalist was written under a DIFFERENT config signature"
            elif _m.get("target") != PRIMARY_TARGET or _m.get("reducer") != PRIMARY_REDUCER:
                _stale = (f"the finalist came from target={_m.get('target')!r}, "
                          f"reducer={_m.get('reducer')!r}, not the primary endpoint")
        except Exception as _e:
            _stale = f"the finalist sidecar is unreadable ({_e})"
    if _stale:
        _need = {k for k in DONE if k[2] == PRIMARY_TARGET and k[3] == PRIMARY_REDUCER}
        if _need:
            print(f"  {_stale}, so the {len(_need)} primary-endpoint cell(s) are recomputed "
                  f"to leave a fitted model behind (nb4 attributes that artefact).")
            _INVALID |= _need
    if _INVALID:
        print(f"  DISCARDING {len(_INVALID)} logged cell(s):")
        for _k in sorted(_INVALID):
            print(f"    {_k[0]} | {_k[1]} | {_k[2]} | {_k[3]}")
        DONE = DONE - _INVALID
        print("  (the new rows supersede the old ones -- load_results keeps the last per key)")

if DONE:
    print(f"{len(DONE)} cell(s) already logged under this signature; they will be skipped.")

FINALIST_STATE = {}
_t_all = time.time()
_targets_for_grid = [PRIMARY_TARGET]

for bid in [b for b in BACKBONES_TO_RUN if b in X_BY_BACKBONE]:
    X = X_BY_BACKBONE[bid]
    for fam in FAMILIES_TO_RUN:
        spatial = QUANTUM_FAMILIES[fam]["spatial"]
        M = M_BY_BACKBONE.get(bid) if spatial else None
        if spatial and M is None:
            print(f"[{bid} | {fam}] SKIPPED: no spatial maps for this backbone "
                  f"(re-run Phase A with SAVE_SPATIAL_MAP=True).")
            continue
        for tgt in _targets_for_grid:
            key = (bid, fam, tgt, PRIMARY_REDUCER)
            if key in DONE:
                print(f"[{bid} | {fam} | {tgt}] already logged -- skipped.")
                continue
            print(f"\n{'='*94}\n[{bid} | {fam} | {tgt} | reducer={PRIMARY_REDUCER}]  "
                  f"{QUANTUM_FAMILIES[fam]['placement']}\n{'='*94}")
            out = run_cell(bid, fam, PRIMARY_REDUCER, tgt, X, TARGETS[tgt], GROUPS_ALL,
                           CFG, M=M, verbose=True, keep_model=True)
            if out is None:
                continue
            rows, state = out
            append_rows(RESULTS_JSONL, rows)
            FINALIST_STATE[key] = state
            print(f"    logged -> {RESULTS_JSONL}")

print(f"\ngrid wall clock this session: {(time.time()-_t_all)/60:.1f} min")

LB = load_results(RESULTS_JSONL, CFG["signature"])
if LB.empty:
    raise SystemExit("the grid produced nothing -- scroll up for the per-cell reason.")
LB.to_csv(RESULTS_CSV, index=False)
print(f"leaderboard: {len(LB)} rows -> {RESULTS_CSV}")

In [ ]:
# 9b. FINALIST SWEEP -- the extra reducers and the exploratory subtypes
#
# The grid above deliberately runs ONE reducer and ONE target: three reducers x six targets
# would be ~18x the compute for the same headline claim. But the reducer choice matters more
# than the ansatz does, so the finalist cell is re-run across all of them here -- and the
# subtype targets, which are exploratory (small positive counts, wide intervals), are run on
# the finalist only.
#
# Set RUN_FINALIST_SWEEP = False to skip.
import time

import numpy as np

RUN_FINALIST_SWEEP = True

# Re-read what is logged. `DONE` was computed before the grid ran, so on a second pass
# through this cell every sweep row already on disk looked new and was recomputed --
# minutes to hours of duplicated work whose only visible symptom is a slow notebook.
DONE = set() if FORCE_RERUN else done_cells(RESULTS_JSONL, CFG["signature"])

if RUN_FINALIST_SWEEP and not LB.empty:
    _q = LB[(LB["arm"] == "quantum") & LB["holdout_auroc"].notna()]
    if _q.empty:
        print("no quantum rows -- nothing to sweep.")
    else:
        _best = _q.sort_values("holdout_auroc", ascending=False).iloc[0]
        _bid, _fam = _best["backbone"], _best["family"]
        print(f"finalist cell: {_bid} | {_fam} "
              f"(holdout AUROC {float(_best['holdout_auroc']):.4f})\n")
        _X = X_BY_BACKBONE[_bid]
        _M = M_BY_BACKBONE.get(_bid) if QUANTUM_FAMILIES[_fam]["spatial"] else None

        _jobs = [(_fam, r, PRIMARY_TARGET) for r in FINALIST_REDUCERS
                 if r != PRIMARY_REDUCER]
        if RUN_SECONDARY_ON_FINALIST_ONLY:
            _jobs += [(_fam, PRIMARY_REDUCER, t) for t in SECONDARY_TARGETS if t in TARGETS]
        print(f"{len(_jobs)} additional run(s): "
              f"{len(FINALIST_REDUCERS)-1} extra reducer(s) + "
              f"{sum(1 for t in SECONDARY_TARGETS if t in TARGETS)} subtype target(s)\n")

        _t0 = time.time()
        for _f2, _red2, _tgt2 in _jobs:
            _key = (_bid, _f2, _tgt2, _red2)
            if _key in DONE:
                print(f"[{_tgt2} | reducer={_red2}] already logged -- skipped.")
                continue
            print(f"\n{'='*94}\n[{_bid} | {_f2} | {_tgt2} | reducer={_red2}]\n{'='*94}")
            _out = run_cell(_bid, _f2, _red2, _tgt2, _X, TARGETS[_tgt2], GROUPS_ALL, CFG,
                            M=_M, verbose=True, keep_model=True)
            if _out is None:
                continue
            append_rows(RESULTS_JSONL, _out[0])
            FINALIST_STATE[_key] = _out[1]
        print(f"\nfinalist sweep: {(time.time()-_t0)/60:.1f} min")
        LB = load_results(RESULTS_JSONL, CFG["signature"])
        LB.to_csv(RESULTS_CSV, index=False)
        print(f"leaderboard now {len(LB)} rows")

        def _ci(v, lo, hi, p=3):
            try:
                v, lo, hi = float(v), float(lo), float(hi)
            except (TypeError, ValueError):
                return "n/a"
            if not np.isfinite(v):
                return "n/a"
            if not (np.isfinite(lo) and np.isfinite(hi)):
                return f"{v:.{p}f}"
            return f"{v:.{p}f} [{lo:.{p}f},{hi:.{p}f}]"

        _sub = LB[(LB["arm"] == "quantum") & (LB["target"] != PRIMARY_TARGET)]
        if not _sub.empty:
            print("\nEXPLORATORY subtype results -- read the INTERVALS, not the point estimates.")
            print("At these positive counts a point AUROC is not a claim.\n")
            print(f"  {'target':<20}{'n pos':>7}{'holdout pos':>13}   {'AUROC [95% CI]':<26}")
            def _i(v):
                # A skipped or degenerate cell writes null here, which json turns back into
                # NaN. int(NaN) raises ValueError and takes the whole results cell with it.
                try:
                    return f"{int(v):,}" if np.isfinite(float(v)) else "n/a"
                except (TypeError, ValueError):
                    return "n/a"

            for _, r in _sub.iterrows():
                print(f"  {r['target']:<20}{_i(r['n_positive']):>7}"
                      f"{_i(r['n_holdout_pos']):>13}   "
                      f"{_ci(r['holdout_auroc'], r['holdout_auroc_lo'], r['holdout_auroc_hi']):<26}")
else:
    print("finalist sweep skipped.")

In [ ]:
# 9c. PERSIST THE FINALIST -- the artefact nb4 attributes through
#
# Without this cell nb4 was a set of instructions: it defined Quantum Integrated Gradients
# and then told you to re-create a trained adapter by hand, in the same session, from
# objects the grid had already thrown away. Here the winning cell is written to disk --
# circuit weights, readout, and the reducer collapsed to an explicit affine map -- so nb4
# reloads it and runs.
#
# THE MAP IS AFFINE, NOT LINEAR. `transform_np` for the PCA reducers is
#     ((Z - pca.mean_) @ components_[keep].T) / scale
# Rebuilding it downstream as a bare matrix multiply drops the `pca.mean_` term. In
# practice that term is small -- StandardScaler has already centred Z, so pca.mean_ is
# near zero on the training data -- but "near zero" is not a property anyone checked, and
# it is not guaranteed for a fold with a different scaler. It is folded into `b` here and
# the whole map is verified against transform_np on real rows before anything is written,
# so nb4 is exercising the function that was actually trained rather than one that
# resembles it.
import json
import os

import numpy as np
import torch


def reducer_affine_map(red):
    """Return (mu, sd, W, b) with pre-tanh angles = ((f - mu) / sd) @ W.T + b.

    One representation for all three reducers, so nb4 never has to branch on which one
    produced the finalist.
    """
    mu = np.asarray(red._sc.mean_, dtype=np.float64)
    sd = np.asarray(red._sc.scale_, dtype=np.float64)
    if red.mode == "trainable":
        W = red.proj.weight.detach().cpu().numpy().astype(np.float64)
        b = red.proj.bias.detach().cpu().numpy().astype(np.float64)
        return mu, sd, W, b
    C = np.asarray(red._pca.components_[red._keep], dtype=np.float64)   # (n_qubits, d)
    s = np.asarray(red._scale, dtype=np.float64).reshape(-1)            # (n_qubits,)
    W = C / s[:, None]
    b = -(C @ np.asarray(red._pca.mean_, dtype=np.float64)) / s
    return mu, sd, W, b


def check_affine_map(red, X, atol=1e-3):
    """Prove the collapsed map reproduces the fitted reducer on real rows before trusting
    it. This is the check that would have caught the missing pca.mean_ term."""
    mu, sd, W, b = reducer_affine_map(red)
    # `ref` must come from the OBJECT THAT WAS TRAINED, by the same route the training loop
    # used, or the check tests the collapsed map against itself and passes unconditionally.
    #   pca / pca_supervised : transform_np already returns the pre-tanh angles
    #   trainable            : transform_np returns standardised features and the
    #                          projection lives in the module's forward, so both are needed
    T = np.asarray(red.transform_np(X), dtype=np.float64)
    if red.mode == "trainable":
        with torch.no_grad():
            ref = red(torch.as_tensor(T, dtype=torch.float32)).cpu().numpy().astype(np.float64)
    else:
        ref = T
    z = (np.asarray(X, dtype=np.float64) - mu) / sd
    got = z @ W.T + b
    err = float(np.max(np.abs(got - ref)))
    rel = err / max(float(np.max(np.abs(ref))), 1e-12)
    if not np.isfinite(err) or rel > atol:
        raise RuntimeError(
            f"the collapsed reducer map disagrees with the fitted reducer by {err:.3e} "
            f"({rel:.2%} of full scale). Do NOT export it: every attribution nb4 produces "
            f"would explain a different function from the one that was trained.")
    return err, rel


FINALIST_PATH = os.path.join(OUTPUT_DIR, "finalist_adapter.pt")

_saved = None
if FINALIST_STATE:
    # Attribution needs the POOLED-VECTOR path: Q3's adapter consumes a (C,d,h,w) feature
    # map through a 1x1x1 convolution, which is a different graph. The best non-spatial
    # cell is exported instead, and said so, rather than exporting something nb4 cannot use.
    # THE FINALIST MUST COME FROM THE CONFIRMATORY GRID.
    #
    # The first run exported B1 | Q1_sel | intraventricular because that cell scored 0.8099
    # against 0.7488 for the pre-specified endpoint -- but `intraventricular` is one of five
    # EXPLORATORY subtypes, and `trainable`/`pca` are exploratory reducers. Taking the max
    # over all of them is selection on exploratory results: nb4 would then attribute a model
    # that was chosen by looking at the numbers it is meant to explain, and the reported
    # AUROC of that cell is a winner's-curse estimate. The finalist is the best cell at the
    # PRIMARY target and PRIMARY reducer; everything else stays exploratory.
    _cands = []
    for (_bid, _fam, _tgt, _red), _st in FINALIST_STATE.items():
        if QUANTUM_FAMILIES[_fam]["spatial"]:
            continue
        if _tgt != PRIMARY_TARGET or _red != PRIMARY_REDUCER:
            continue
        if not _st or not _st.get("fitted") or _st["fitted"].get("model") is None:
            continue
        _r = LB[(LB["backbone"] == _bid) & (LB["family"] == _fam) & (LB["target"] == _tgt)
                & (LB["reducer"] == _red) & (LB["arm"] == "quantum")]
        _au = float(_r["holdout_auroc"].iloc[0]) if len(_r) else float("nan")
        if np.isfinite(_au):
            _cands.append((_au, (_bid, _fam, _tgt, _red), _st))
    if not _cands:
        print(f"no non-spatial quantum cell at target={PRIMARY_TARGET!r}, "
              f"reducer={PRIMARY_REDUCER!r} with a fitted model -- nothing to export.")
        print("  (Q3 is a deep-placement arm; its attribution graph is not this one.)")
    else:
        _cands.sort(key=lambda t: -t[0])
        _au, (_bid, _fam, _tgt, _red), _st = _cands[0]
        _model = _st["fitted"]["model"]
        _reducer = _st["fitted"]["reducer"]
        _X = X_BY_BACKBONE[_bid]
        _err, _rel = check_affine_map(_reducer, _X[_st["pool_idx"]][:256])
        _mu, _sd, _W, _b = reducer_affine_map(_reducer)
        _hold_series = [str(s) for s in np.asarray(SERIES_ALL)[_st["hold_idx"]]]
        _payload = {
            "backbone": _bid, "family": _fam, "target": _tgt, "reducer": _red,
            "holdout_auroc": _au, "config": CFG,
            "n_qubits": int(CFG["n_qubits"]), "depth": int(CFG["depth"]),
            "feature_dim": int(_X.shape[1]),
            "model_state_dict": {k: v.detach().cpu()
                                 for k, v in _model.state_dict().items()
                                 if not k.startswith("reducer.")},
            "reducer_mu": _mu.astype(np.float32), "reducer_sd": _sd.astype(np.float32),
            "reducer_W": _W.astype(np.float32), "reducer_b": _b.astype(np.float32),
            "reducer_mode": _reducer.mode,
            "affine_check_max_abs_err": _err, "affine_check_rel_err": _rel,
            "holdout_series": _hold_series,
            "holdout_y": np.asarray(_st["yh"]).astype(int).tolist(),
            "holdout_p": np.asarray(_st["hold_p"]["quantum"], dtype=float).tolist(),
        }
        torch.save(_payload, FINALIST_PATH)
        # A sidecar recording WHICH cell produced this file. Without it a stale artefact
        # from an earlier grid is indistinguishable from a current one, and nb4 would
        # attribute a model that no longer corresponds to any row in the leaderboard.
        with open(os.path.join(OUTPUT_DIR, "finalist_adapter.meta.json"), "w") as _fh:
            json.dump({"config_sig": CFG["signature"], "backbone": _bid, "family": _fam,
                       "target": _tgt, "reducer": _red, "holdout_auroc": float(_au)},
                      _fh, indent=2)
        _saved = FINALIST_PATH
        print(f"finalist exported -> {FINALIST_PATH}")
        print(f"  cell        : {_bid} | {_fam} | {_tgt} | reducer={_red}")
        print(f"  holdout AUROC {_au:.4f}   feature dim {_X.shape[1]}   "
              f"{CFG['n_qubits']} qubits x depth {CFG['depth']}")
        print(f"  reducer collapsed to an affine map, verified against transform_np to "
              f"{_rel:.2e} relative")
        print(f"  {len(_hold_series):,} held-out series recorded, so nb4 attributes a case "
              f"the adapter never trained on")
else:
    print("FINALIST_STATE is empty -- run the grid cell above first.")
    print("(Resuming a fully-logged grid skips every cell and therefore keeps no models;")
    print(" set FORCE_RERUN = True for one pass if you need this artefact.)")


In [ ]:
# 10. RESULTS (1) -- discrimination, with patient-clustered bootstrap intervals
#
# AUPRC is printed next to the prevalence, because a random ranker scores exactly the
# prevalence: an AUPRC of 0.30 is excellent at 3% and worthless at 40%.
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)


def _f(v, w=0, p=3, sign=""):
    try:
        x = float("nan") if v is None else float(v)
    except (TypeError, ValueError):
        x = float("nan")
    return f"{'n/a':>{w}}" if not np.isfinite(x) else f"{x:>{sign}{w}.{p}f}"


def _ci(v, lo, hi, p=3):
    try:
        v, lo, hi = float(v), float(lo), float(hi)
    except (TypeError, ValueError):
        return "n/a"
    if not np.isfinite(v):
        return "n/a"
    if not (np.isfinite(lo) and np.isfinite(hi)):
        return f"{v:.{p}f}"
    return f"{v:.{p}f} [{lo:.{p}f},{hi:.{p}f}]"


L = LB.copy()
L["arm"] = pd.Categorical(L["arm"], ["quantum", "classical", "linear_probe"], ordered=True)
L = L.sort_values(["target", "backbone", "family", "arm"])

print("=" * 124)
print("(1) DISCRIMINATION on held-out patients   [point estimate, 95% patient-clustered CI]")
print("=" * 124)
for tgt, g0 in L.groupby("target", sort=False):
    print(f"\n### target = {tgt}   n={int(g0['n_total'].iloc[0]):,}  "
          f"positives={int(g0['n_positive'].iloc[0]):,}  "
          f"prevalence={float(g0['prevalence'].iloc[0]):.2%}   "
          f"holdout n={int(g0['n_holdout'].iloc[0]):,} ({int(g0['n_holdout_pos'].iloc[0])} pos)")
    for bid, g in g0.groupby("backbone", sort=False):
        print(f"\n  {bid}")
        print(f"    {'family':<14}{'arm':<14}{'params':>8}{'tail':>6}  {'cvAUROC':>8}   "
              f"{'holdout AUROC':<24}{'holdout AUPRC':<24}{'lift':>6}")
        for _, r in g.iterrows():
            print(f"    {str(r['family']):<14}{str(r['arm']):<14}"
                  f"{int(r['trainable_params']):>8,}{int(r['adapter_tail_params']):>6}  "
                  f"{_f(r['cv_auroc'], 8, 4)}   "
                  f"{_ci(r['holdout_auroc'], r['holdout_auroc_lo'], r['holdout_auroc_hi']):<24}"
                  f"{_ci(r['holdout_auprc'], r['holdout_auprc_lo'], r['holdout_auprc_hi']):<24}"
                  f"{_f(r['holdout_auprc_lift'], 6, 1)}x")
    print(f"\n  (AUPRC baseline = prevalence = "
          f"{_f(g0['holdout_auprc_baseline'].iloc[0], 0, 3)}; 'lift' is AUPRC / baseline)")

In [ ]:
# 11. RESULTS (2) -- the comparison that IS the result
import numpy as np
import pandas as pd

#
# A quantum AUROC on its own is not a finding. A ~40-parameter circuit against a
# ~40-parameter MLP fed the identical reduced input is the whole comparison, and a null
# result here is a real result.
print("=" * 118)
print("(2) QUANTUM vs ITS PARAMETER-MATCHED CLASSICAL CONTROL   (paired patient bootstrap)")
print("=" * 118)
Q = L[L["arm"] == "quantum"]
print(f"  {'backbone':<26}{'family':<14}{'dAUROC (q - matched classical)':<34}{'p':>8}   verdict")
for _, r in Q.iterrows():
    d, lo, hi, p = (r.get("vs_classical_diff"), r.get("vs_classical_lo"),
                    r.get("vs_classical_hi"), r.get("vs_classical_p_two_sided"))
    try:
        d, p = float(d), float(p)
    except (TypeError, ValueError):
        continue
    if not np.isfinite(d):
        continue
    verdict = ("no interval" if not np.isfinite(p) else
               "no detectable difference" if p >= 0.05 else
               "quantum ahead" if d > 0 else "classical ahead")
    cell = f"{d:+.4f} [{float(lo):+.4f}, {float(hi):+.4f}]"
    print(f"  {r['backbone']:<26}{str(r['family']):<14}{cell:<34}{_f(p, 8, 4)}   {verdict}")

print()
print(f"  {'backbone':<26}{'family':<14}{'dAUROC (q - linear probe)':<34}{'p':>8}")
for _, r in Q.iterrows():
    d, lo, hi, p = (r.get("vs_probe_diff"), r.get("vs_probe_lo"),
                    r.get("vs_probe_hi"), r.get("vs_probe_p_two_sided"))
    try:
        d = float(d)
    except (TypeError, ValueError):
        continue
    if not np.isfinite(d):
        continue
    cell = f"{d:+.4f} [{float(lo):+.4f}, {float(hi):+.4f}]"
    print(f"  {r['backbone']:<26}{str(r['family']):<14}{cell:<34}{_f(p, 8, 4)}")
print()
print("  The linear probe is the floor: the full frozen vector, no adapter, no reduction.")
print("  An adapter that does not beat it is buying nothing with its dimensionality")
print("  reduction, and the REDUCER -- not the ansatz -- is where to look.")

print()
print("=" * 118)
print("(3) OPERATING POINTS -- thresholds chosen on out-of-fold predictions, applied unchanged")
print("=" * 118)
for tgt, g0 in L.groupby("target", sort=False):
    print(f"\n{tgt}")
    print(f"  {'backbone':<24}{'family':<12}{'arm':<12}{'point':<16}{'ACC':>7}{'vs maj':>8}"
          f"{'BAL':>7}   {'sensitivity':<20}{'specificity':<20}")
    for _, r in g0.iterrows():
        for pt in ("youden", "high_sensitivity"):
            # NB `float(v or nan)` would be wrong here: an accuracy of exactly 0.0 is
            # falsy and would be silently reported as missing.
            _acc = r.get(f"{pt}__accuracy")
            if _acc is None or not np.isfinite(pd.to_numeric(_acc, errors="coerce")):
                continue
            lbl = pt if pt != "high_sensitivity" else f"high-sens@{TARGET_SENSITIVITY:.2f}"
            sens = _ci(r[f"{pt}__sensitivity"], r[f"{pt}__sensitivity_lo"], r[f"{pt}__sensitivity_hi"])
            spec = _ci(r[f"{pt}__specificity"], r[f"{pt}__specificity_lo"], r[f"{pt}__specificity_hi"])
            print(f"  {r['backbone']:<24}{str(r['family']):<12}{str(r['arm']):<12}{lbl:<16}"
                  f"{_f(r[f'{pt}__accuracy'],7)}{_f(r[f'{pt}__accuracy_minus_majority'],8,3,'+')}"
                  f"{_f(r[f'{pt}__balanced_accuracy'],7)}   {sens:<20}{spec:<20}")
print()
print("  'vs maj' is accuracy minus the majority-class rate -- the only column that answers")
print("  whether the model is doing anything at all. Sensitivity/specificity CIs are Wilson.")

In [ ]:
# 12. RQ1 -- which factor actually drives generalisation, and BH across the grid
#
# Naming whichever single cell topped the leaderboard is a weak, overfitting-prone claim.
# The useful question is at the FAMILY level: does a family generalise better regardless of
# which backbone it is paired with? That is what the grid is for.
print("=" * 100)
print("RQ1 -- two-way variance decomposition over the grid")
print("=" * 100)
GRID = primary_grid(LB, PRIMARY_REDUCER, PRIMARY_TARGET)
_nq = int((GRID["arm"] == "quantum").sum())
print(f"  confirmatory grid: {_nq} quantum cell(s) at reducer={PRIMARY_REDUCER!r}, "
      f"target={PRIMARY_TARGET!r}")
if len(LB) > len(GRID):
    print(f"  ({len(LB)-len(GRID)} further row(s) from the finalist sweep are EXCLUDED here "
          f"and reported separately below)")
DEC = family_variance_decomposition(GRID, metric="holdout_auroc", verbose=True)
print()
DEC_CV = family_variance_decomposition(GRID, metric="cv_auroc", verbose=False)
if DEC_CV:
    print(f"  (same decomposition on cross-validated AUROC: backbone "
          f"{DEC_CV['frac_backbone']:.1%} | family {DEC_CV['frac_family']:.1%} | "
          f"interaction {DEC_CV['frac_interaction']:.1%})")
print()
print("  Read this before any single cell: if 'quantum family' explains little of the")
print("  variance and 'backbone' explains most, then the ansatz is not what is driving the")
print("  result -- and that is a finding, not a disappointment.")

print()
print("=" * 100)
print("MULTIPLICITY -- Benjamini-Hochberg across every quantum-vs-control comparison")
print("=" * 100)
# TWO families of tests, corrected separately. Pooling the pre-specified grid with the
# exploratory sweep into one BH family penalises the confirmatory result for questions it
# never asked, and lets an exploratory hit borrow the grid's credibility. Which comparisons
# were pre-specified is a property of the design, not of the p-values.
print("CONFIRMATORY -- the pre-specified grid "
      f"(reducer={PRIMARY_REDUCER!r}, target={PRIMARY_TARGET!r})")
BH = grid_multiplicity_correction(GRID, alpha=0.05, verbose=True)
_sweep = LB[~LB.index.isin(GRID.index)]
if not _sweep.empty:
    print()
    print(f"EXPLORATORY -- the finalist sweep ({len(_sweep)} row(s): other reducers and the")
    print("subtype targets). Corrected as its own family and reported as exploratory.")
    BH_SWEEP = grid_multiplicity_correction(_sweep, alpha=0.05, verbose=True)
else:
    BH_SWEEP = None
print()
print("  Without this correction, the best-looking cell of N is a selection effect rather")
print("  than a finding. With a deliberately small grid the correction costs little power,")
print("  which is one of the reasons the grid is small.")

In [ ]:
# 13. NISQ robustness -- what survives a noisy device?
#
# The whole grid is statevector simulation: a noiseless idealisation. The reviewer question
# this answers is "would this survive real hardware?". The standard cheap answer: keep the
# TRAINED angles exactly as they are, re-evaluate on a density-matrix simulator with a
# depolarizing channel after every entangling gate, and report how much AUROC is retained.
#
# Two things this deliberately does NOT do, both of which would overstate the result:
#   * it does not RETRAIN under noise -- noise-aware training would do better and would not
#     answer the question, which is whether the circuit you trained transfers.
#   * it does not model readout error, T1/T2, or a device's connectivity graph. A uniform
#     two-qubit depolarizing channel is the conventional first-order proxy. Say that.
#
# A citable bar: retaining >90% of the noise-free AUROC supports a near-term-viability
# claim. Retaining less is still worth reporting rather than omitting.
import numpy as np
import pandas as pd
import torch

Qrows = LB[(LB["arm"] == "quantum") & LB["holdout_auroc"].notna()]
if Qrows.empty:
    print("no quantum rows to make noisy.")
else:
    best = Qrows.sort_values("holdout_auroc", ascending=False).iloc[0]
    print(f"finalist: {best['backbone']} | {best['family']} | {best['target']} | "
          f"reducer={best.get('reducer', PRIMARY_REDUCER)}  "
          f"(noise-free holdout AUROC {float(best['holdout_auroc']):.4f})\n")
    bid, fam, tgt = best["backbone"], best["family"], best["target"]
    if QUANTUM_FAMILIES[fam]["spatial"]:
        print("the finalist is the spatial (Q3) arm; the noise ablation below is run on the")
        print("best NON-spatial family instead, because a per-position density-matrix")
        print("simulation over every held-out volume is not a cheap ablation.")
        alt = Qrows[~Qrows["family"].isin([f for f in QUANTUM_FAMILIES
                                           if QUANTUM_FAMILIES[f]["spatial"]])]
        if alt.empty:
            raise SystemExit("no non-spatial quantum arm available for the NISQ ablation.")
        best = alt.sort_values("holdout_auroc", ascending=False).iloc[0]
        bid, fam, tgt = best["backbone"], best["family"], best["target"]
        print(f"  -> using {bid} | {fam}\n")

    # REFIT WITH THE FINALIST'S OWN REDUCER, not PRIMARY_REDUCER. The finalist can come
    # from the sweep, which runs other reducers -- and refitting with a different one makes
    # the "noise-free holdout AUROC" printed above belong to a different model from the one
    # in the table below. With reducer="trainable" it does not even mismatch quietly: the
    # projection lives inside the module, so transform_np returns the full standardised
    # feature vector and AngleEmbedding is handed 2,304 values for N_QUBITS wires.
    red_used = str(best.get("reducer", PRIMARY_REDUCER))
    if red_used != PRIMARY_REDUCER:
        print(f"  (the finalist used reducer={red_used!r}; refitting with that one)")
    X = X_BY_BACKBONE[bid]
    y = TARGETS[tgt]
    pool, hold = make_holdout_split(y, GROUPS_ALL, HOLDOUT_FRAC, SEED)
    p_clean, _, _, extra = run_arm("quantum", X[pool], y[pool], X[hold], fam,
                                   red_used, CFG, SEED, return_model=True)
    model = extra["model"]
    red = extra["reducer_obj"]
    # The pre-tanh values, by the route the model itself uses: transform_np already yields
    # them for the PCA reducers, while the trainable one keeps its projection in the module.
    _pre = red.transform_np(X[hold])
    if red.mode == "trainable":
        with torch.no_grad():
            _pre = red(torch.as_tensor(_pre, dtype=torch.float32)).cpu().numpy()
    ang = (np.pi / 2.0) * np.tanh(_pre)
    if ang.shape[1] != N_QUBITS:
        raise RuntimeError(f"the angles are {ang.shape[1]}-d but the circuit has "
                           f"{N_QUBITS} wires -- the reducer was not applied.")
    W = model.q.weights.detach().double()
    out_w = model.out.weight.detach().double()
    out_b = model.out.bias.detach().double()

    print(f"{'p_depol':>9}{'AUROC':>9}{'retained':>10}{'AUPRC':>9}{'retained':>10}")
    print("-" * 48)
    rows_nisq = []
    for p_dep in [0.0, NISQ_P, 2 * NISQ_P, 5 * NISQ_P]:
        z = nisq_noisy_expvals(fam, W, torch.as_tensor(ang, dtype=torch.float64),
                               p_dep, N_QUBITS, DEPTH)
        logit = (z @ out_w.T).squeeze(-1) + out_b
        p_noisy = torch.sigmoid(logit).numpy()
        au, ap = fast_auroc(y[hold], p_noisy), fast_auprc(y[hold], p_noisy)
        if p_dep == 0.0:
            au0, ap0 = au, ap
        rows_nisq.append({"p_depol": p_dep, "auroc": au, "auprc": ap,
                          "auroc_retained": au / au0 if au0 else np.nan,
                          "auprc_retained": ap / ap0 if ap0 else np.nan})
        print(f"{p_dep:>9.3f}{au:>9.4f}{au/au0*100:>9.1f}%{ap:>9.4f}{ap/ap0*100:>9.1f}%")
    print()
    print("  p=0 is the density-matrix simulator with no noise. It should reproduce the")
    print("  statevector AUROC almost exactly -- if it does not, the rebuilt ansatz does not")
    print("  match the trained one and this table is measuring the wrong circuit.")
    _chk = abs(rows_nisq[0]["auroc"] - float(best["holdout_auroc"]))
    print(f"  |p=0 AUROC - statevector AUROC| = {_chk:.4f} "
          f"({'consistent' if _chk < 0.02 else 'CHECK THIS -- larger than expected'})")
    r1 = rows_nisq[1]["auroc_retained"]
    print(f"\n  at p={NISQ_P}: {r1*100:.1f}% of the noise-free AUROC retained "
          f"({'supports' if r1 > 0.90 else 'does NOT support'} a >90% near-term-viability claim)")
    pd.DataFrame(rows_nisq).to_csv(os.path.join(OUTPUT_DIR, "nisq_ablation.csv"), index=False)

In [ ]:
# 14. Figures -- ROC and precision-recall on the held-out patients
#
# The PR panel carries a dashed baseline at the prevalence, because that is exactly what a
# random ranker scores and a PR curve without it is unreadable at these prevalences.
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_recall_curve, roc_curve

_cells = [(b, f) for b in X_BY_BACKBONE for f in FAMILIES_TO_RUN
          if not (QUANTUM_FAMILIES[f]["spatial"] and b not in M_BY_BACKBONE)]
if not _cells:
    print("nothing to plot.")
else:
    tgt = PRIMARY_TARGET
    y = TARGETS[tgt]
    pool, hold = make_holdout_split(y, GROUPS_ALL, HOLDOUT_FRAC, SEED)
    yh = y[hold]
    # This split is RECOMPUTED, while the predictions below come out of FINALIST_STATE,
    # which the grid computed with cfg["holdout_frac"] and cfg["seed"]. Edit HOLDOUT_FRAC
    # or SEED after running the grid and the two disagree -- at which point every curve is
    # drawn from cached predictions against the wrong labels, and it still looks like a
    # perfectly ordinary ROC. Checked against a split the grid actually stored.
    for _k, _st in FINALIST_STATE.items():
        if _st and _st.get("hold_idx") is not None and len(_st["hold_idx"]) and \
                not np.array_equal(np.asarray(_st["hold_idx"]), np.asarray(hold)):
            raise RuntimeError(
                f"the holdout split recomputed here ({len(hold)} rows) is not the one the "
                f"grid used for {_k} ({len(_st['hold_idx'])} rows). HOLDOUT_FRAC or SEED "
                f"changed since the grid ran -- re-run the grid with FORCE_RERUN = True, "
                f"or these curves plot cached predictions against the wrong labels.")
        break
    nrow = len(X_BY_BACKBONE)
    fig, axes = plt.subplots(nrow, 2, figsize=(13, 4.8 * nrow), squeeze=False)
    for i, bid in enumerate(X_BY_BACKBONE):
        X = X_BY_BACKBONE[bid]
        M = M_BY_BACKBONE.get(bid)
        ax0, ax1 = axes[i][0], axes[i][1]
        for fam in FAMILIES_TO_RUN:
            if QUANTUM_FAMILIES[fam]["spatial"] and M is None:
                continue
            key = (bid, fam, tgt, PRIMARY_REDUCER)
            if key in FINALIST_STATE:
                p = FINALIST_STATE[key]["hold_p"]["quantum"]
            else:
                p, *_ = run_arm("quantum", X[pool], y[pool], X[hold], fam, PRIMARY_REDUCER,
                                CFG, SEED,
                                Mtr=(M[pool] if M is not None else None),
                                Mte=(M[hold] if M is not None else None))
            fpr, tpr, _ = roc_curve(yh, p)
            ax0.plot(fpr, tpr, lw=1.7, label=f"{fam}  {fast_auroc(yh, p):.3f}")
            pr, rc, _ = precision_recall_curve(yh, p)
            ax1.plot(rc, pr, lw=1.7, label=f"{fam}  {fast_auprc(yh, p):.3f}")
        # the two controls, from the primary family's cell
        key = (bid, FAMILIES_TO_RUN[0], tgt, PRIMARY_REDUCER)
        if key in FINALIST_STATE:
            for arm, style in (("classical", "--"), ("linear_probe", ":")):
                p = FINALIST_STATE[key]["hold_p"][arm]
                fpr, tpr, _ = roc_curve(yh, p)
                ax0.plot(fpr, tpr, style, lw=1.4, c="k", alpha=0.65,
                         label=f"{arm}  {fast_auroc(yh, p):.3f}")
                pr, rc, _ = precision_recall_curve(yh, p)
                ax1.plot(rc, pr, style, lw=1.4, c="k", alpha=0.65,
                         label=f"{arm}  {fast_auprc(yh, p):.3f}")
        ax0.plot([0, 1], [0, 1], "k-", lw=0.7, alpha=0.35)
        ax0.set_xlabel("1 - specificity"); ax0.set_ylabel("sensitivity")
        ax0.set_title(f"{bid}\nROC -- {tgt}, {len(yh)} held-out series "
                      f"({int(yh.sum())} pos)", fontsize=10)
        ax0.legend(fontsize=7.5, loc="lower right"); ax0.grid(alpha=0.25)
        ax1.axhline(float(yh.mean()), ls="--", c="k", lw=0.8, alpha=0.5,
                    label=f"prevalence {yh.mean():.3f}")
        ax1.set_ylim(0, 1)
        ax1.set_xlabel("recall"); ax1.set_ylabel("precision")
        ax1.set_title(f"{bid}\nprecision-recall -- {tgt}", fontsize=10)
        ax1.legend(fontsize=7.5, loc="upper right"); ax1.grid(alpha=0.25)
    fig.tight_layout()
    _p = os.path.join(FIG_DIR, "phaseb_curves.png")
    fig.savefig(_p, dpi=150, bbox_inches="tight")
    print(f"saved {_p}")
    plt.show()

In [ ]:
# 15. Export -- the tables the manuscript actually needs
import numpy as np
import pandas as pd

# `reducer` was missing from this list, and without it the finalist sweep's rows are
# indistinguishable from the grid's: the same (backbone, family, target) appearing three
# times with different numbers and nothing to say why. Anyone grouping this CSV by the
# columns it does carry would silently average three different experiments.
MAIN_COLS = ["backbone", "family", "target", "reducer", "arm", "trainable_params",
             "adapter_tail_params", "cv_auroc", "holdout_auroc", "holdout_auroc_lo",
             "holdout_auroc_hi", "holdout_auprc", "holdout_auprc_lo", "holdout_auprc_hi",
             "holdout_auprc_baseline", "vs_classical_diff", "vs_classical_lo",
             "vs_classical_hi", "vs_classical_p_two_sided", "brier", "ece"]

# TABLE 1 IS THE CONFIRMATORY GRID ONLY. Built from LB, it carried the finalist sweep too
# -- and since the table shows Backbone and Family but not Reducer or Target, that printed
# as four rows reading "B3 resnet18 medicalnet & Q2 reupload" with four different AUROCs
# and no way to tell them apart. A reviewer reads that as an error in the table, and they
# are not wrong.
_grid = primary_grid(LB, PRIMARY_REDUCER, PRIMARY_TARGET)
main = _grid[[c for c in MAIN_COLS if c in _grid.columns]].copy()
main.to_csv(os.path.join(OUTPUT_DIR, "table1_main_results.csv"), index=False)

_sweep_rows = LB[~LB.index.isin(_grid.index)]
if not _sweep_rows.empty:
    _sw = _sweep_rows[[c for c in MAIN_COLS if c in _sweep_rows.columns]].copy()
    _sw.to_csv(os.path.join(OUTPUT_DIR, "table3_finalist_sweep.csv"), index=False)


def latex_main(df):
    q = df[df["arm"] == "quantum"].sort_values(["backbone", "family"])
    dup = q.duplicated(subset=["backbone", "family"], keep=False)
    if dup.any():
        # The columns below cannot distinguish these rows, so refuse rather than print an
        # ambiguous table that looks fine.
        raise ValueError(
            f"{int(dup.sum())} row(s) share a (backbone, family) and would print "
            f"identically: {q.loc[dup, ['backbone','family','target','reducer']].to_dict('records')}. "
            f"Pass primary_grid(LB).")
    out = [r"\begin{tabular}{llrrrr}", r"\toprule",
           r"Backbone & Family & Tail par. & AUROC (95\% CI) & AUPRC & $\Delta$AUROC vs control \\",
           r"\midrule"]
    for _, r in q.iterrows():
        def g(k):
            try:
                v = float(r.get(k))
                return v if np.isfinite(v) else float("nan")
            except (TypeError, ValueError):
                return float("nan")
        au, lo, hi = g("holdout_auroc"), g("holdout_auroc_lo"), g("holdout_auroc_hi")
        ap, d, p = g("holdout_auprc"), g("vs_classical_diff"), g("vs_classical_p_two_sided")
        star = "" if not np.isfinite(p) else (r"$^{*}$" if p < 0.05 else "")
        out.append(f"{str(r['backbone']).replace('_', ' ')} & "
                   f"{str(r['family']).replace('_', ' ')} & "
                   f"{int(r['adapter_tail_params'])} & "
                   f"{au:.3f} [{lo:.3f}, {hi:.3f}] & {ap:.3f} & "
                   f"{d:+.3f}{star} \\\\")
    out += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(out)


tex = latex_main(_grid)
with open(os.path.join(OUTPUT_DIR, "table1_main_results.tex"), "w") as f:
    f.write(tex)
print(tex)

if DEC:
    pd.DataFrame([{
        "metric": DEC["metric"], "n_cells": DEC["n_cells"],
        "frac_backbone": DEC["frac_backbone"], "frac_family": DEC["frac_family"],
        "frac_interaction": DEC["frac_interaction"], "frac_residual": DEC["frac_residual"],
    }]).to_csv(os.path.join(OUTPUT_DIR, "table2_rq1_decomposition.csv"), index=False)

print(f"\nTable 1 is the confirmatory grid: {len(main)} row(s) at "
      f"reducer={PRIMARY_REDUCER!r}, target={PRIMARY_TARGET!r}.")
if not _sweep_rows.empty:
    print(f"The finalist sweep's {len(_sweep_rows)} row(s) are in table3_finalist_sweep.csv "
          f"and are EXPLORATORY -- other reducers and subtype targets, one cell only.")
print(f"\nwrote:")
for f in ["table1_main_results.csv", "table1_main_results.tex",
          "table2_rq1_decomposition.csv", "table3_finalist_sweep.csv",
          "nisq_ablation.csv", "phaseb_leaderboard.csv"]:
    p = os.path.join(OUTPUT_DIR, f)
    print(f"  {'OK ' if os.path.exists(p) else '-- '}{p}")

---

## What to read, in what order

1. **Table (2), quantum vs the parameter-matched control.** This is the result. Everything
   above it is context. A ~40-parameter circuit against a ~40-parameter MLP fed the identical
   reduced input is the entire comparison, and if they are indistinguishable, *that is the
   finding* -- report it rather than hunting for a configuration that flatters the circuit.
2. **The linear-probe row.** If no adapter beats the probe, the reducer is the bottleneck,
   not the ansatz, and the paper's story is about the information bottleneck.
3. **RQ1's decomposition.** If "backbone" explains most of the variance and "family"
   almost none, the ansatz is not what drives generalisation. Also a finding.
4. **The NISQ table**, for the hardware-viability paragraph.

## Next

* `nb3_rq4_2d_vs_3d.ipynb` -- does 3D volumetric context actually earn its cost, or does a
  far cheaper 2D pipeline with the same adapter get most of the way there?
* `nb4_explainability_qig.ipynb` -- quantum integrated gradients, validated against
  ground-truth lesion masks rather than by eye.
